# [Baseline] RandomForest — 풍력발전량 예측

기상 예보 데이터(LDAPS/GFS)로 KPX 발전그룹 3곳의 시간대별 **풍력발전량(kWh)** 을 예측하는 회귀 베이스라인입니다.

- **입력**: 예측 시각별 LDAPS/GFS 기상 예보 격자 평균값 + 시간·요일·계절성 캘린더 변수
- **출력**: `kpx_group_1/2/3` 세 발전그룹의 시간대별 예측 발전량(kWh)
- **모델**: 그룹별 `RandomForestRegressor` (Label 제공 기간이 그룹마다 다르므로 각각 따로 학습)

이 대회는 예측 결과 CSV(`baseline_submit.csv`)를 제출하는 방식이라, 데이터 로드부터 제출 파일 생성까지
이 노트북 하나로 처리합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. 라이브러리 불러오기

데이터 처리(pandas, numpy)와 회귀 모델 학습(scikit-learn)에 필요한 라이브러리를 불러옵니다.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
import os
import random

In [ ]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)  # 주의: 이건 인터프리터 시작 전에 적용돼야 완전히 효과 있음
random.seed(SEED)
np.random.seed(SEED)

## 2. 데이터 로드

발전량 Label(`train_labels.csv`), 제출 양식(`sample_submission.csv`), LDAPS/GFS 기상 예보 데이터를 불러오고,
날짜 컬럼을 datetime으로 변환합니다. `CAPACITY_KWH` 는 그룹별 설비 용량으로, 이후 예측값을 클리핑하는 데
사용합니다.


In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/비타민/풍력발전량예측/data")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {
    "kpx_group_1": 21600,
    "kpx_group_2": 21600,
    "kpx_group_3": 21000,
}

train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")

ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")

train_labels["kst_dtm"] = pd.to_datetime(train_labels["kst_dtm"])
sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

print("train_labels:", train_labels.shape)
print("sample_submission:", sample_submission.shape)
print("ldaps_train:", ldaps_train.shape, "gfs_train:", gfs_train.shape)
print("ldaps_test:", ldaps_test.shape, "gfs_test:", gfs_test.shape)


train_labels: (26304, 4)
sample_submission: (8760, 5)
ldaps_train: (420864, 35) gfs_train: (236736, 40)
ldaps_test: (140160, 35) gfs_test: (78840, 40)


## 3. Feature 생성

LDAPS/GFS는 하나의 예측 시각에 여러 격자가 존재하므로, `forecast_kst_dtm`별로 수치형 기상변수의 평균값을 계산합니다.

추가로 월, 일, 시간, 요일, 주말 여부, 시간/월의 주기성을 나타내는 sin-cos feature를 생성합니다.


In [ ]:
def aggregate_weather(df, prefix):
    df = df.copy()
    df["forecast_kst_dtm"] = pd.to_datetime(df["forecast_kst_dtm"])
    drop_cols = {"data_available_kst_dtm", "grid_id", "latitude", "longitude"}
    value_cols = [c for c in df.columns if c not in {"forecast_kst_dtm", *drop_cols}]
    agg = df.groupby("forecast_kst_dtm")[value_cols].mean()
    agg.columns = [f"{prefix}_{c}_mean" for c in agg.columns]
    return agg.reset_index()


def calendar_features(dt_series):
    dt = pd.to_datetime(dt_series)
    out = pd.DataFrame(index=dt.index)
    out["month"] = dt.dt.month
    out["day"] = dt.dt.day
    out["hour"] = dt.dt.hour
    out["dayofweek"] = dt.dt.dayofweek
    out["is_weekend"] = dt.dt.dayofweek.isin([5, 6]).astype(int)
    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
    out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
    return out


train_weather = aggregate_weather(ldaps_train, "ldaps").merge(
    aggregate_weather(gfs_train, "gfs"), on="forecast_kst_dtm", how="inner"
)
test_weather = aggregate_weather(ldaps_test, "ldaps").merge(
    aggregate_weather(gfs_test, "gfs"), on="forecast_kst_dtm", how="inner"
)

train_base = train_labels.rename(columns={"kst_dtm": "forecast_kst_dtm"})
train_df = train_base.merge(train_weather, on="forecast_kst_dtm", how="left")
test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].merge(
    test_weather, on="forecast_kst_dtm", how="left"
)

X_train = pd.concat(
    [calendar_features(train_df["forecast_kst_dtm"]), train_df.drop(columns=["forecast_kst_dtm", *TARGET_COLS])],
    axis=1,
)
X_test = pd.concat(
    [calendar_features(test_df["forecast_kst_dtm"]), test_df.drop(columns=["forecast_id", "forecast_kst_dtm"])],
    axis=1,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


X_train: (26304, 74)
X_test: (8760, 74)


## 4. 검증 metric

In [ ]:
def metric(answer_df, pred_df):
    group_nmae = []
    group_ficr = []
    for col in TARGET_COLS:
        actual = answer_df[col].to_numpy(dtype=float)
        forecast = pred_df[col].to_numpy(dtype=float)
        capacity = CAPACITY_KWH[col]
        valid = actual >= capacity * 0.10
        actual = actual[valid]
        forecast = forecast[valid]
        error_rate = np.abs(forecast - actual) / capacity
        group_nmae.append(np.mean(error_rate))
        unit_price = np.select(
            [error_rate <= 0.06, error_rate <= 0.08],
            [4.0, 3.0],
            default=0.0,
        )
        earned_settlement = np.sum(actual * unit_price)
        max_settlement = np.sum(actual * 4.0)
        group_ficr.append(earned_settlement / max_settlement)
    one_minus_nmae = 1 - np.mean(group_nmae)
    ficr = np.mean(group_ficr)
    total_score = 0.5 * one_minus_nmae + 0.5 * ficr
    return total_score, one_minus_nmae, ficr


# ── Walk-forward fold 정의 ───────────────────────────
# 세 그룹 라벨이 모두 존재하는 2023~2024 구간에서
# expanding window 방식으로 2개 fold 구성
FOLDS = [
    {
        "name": "fold1_2024H1",
        "train_end": pd.Timestamp("2024-01-01 00:00:00"),   # 이 시각 이전까지 학습
        "valid_start": pd.Timestamp("2024-01-01 01:00:00"),
        "valid_end": pd.Timestamp("2024-07-01 00:00:00"),
    },
    {
        "name": "fold2_2024H2",
        "train_end": pd.Timestamp("2024-07-01 00:00:00"),
        "valid_start": pd.Timestamp("2024-07-01 01:00:00"),
        "valid_end": pd.Timestamp("2025-01-01 00:00:00"),
    },
]

In [ ]:
def run_cv(train_df, weather_cols):
    """
    train_df: forecast_kst_dtm, TARGET_COLS, weather_cols를 포함하는 전체 학습 데이터프레임
    weather_cols: calendar_features를 제외한, 실제 train_df에 존재하는 기상 피처 컬럼 리스트
    """
    fold_scores = []

    for fold in FOLDS:
        tr_mask = train_df["forecast_kst_dtm"] <= fold["train_end"]
        va_mask = (train_df["forecast_kst_dtm"] >= fold["valid_start"]) & \
                  (train_df["forecast_kst_dtm"] <= fold["valid_end"])

        tr_df = train_df.loc[tr_mask].reset_index(drop=True)
        va_df = train_df.loc[va_mask].reset_index(drop=True)

        # calendar 피처는 여기서 직접 생성 (train_df엔 없으므로)
        tr_cal = calendar_features(tr_df["forecast_kst_dtm"])
        va_cal = calendar_features(va_df["forecast_kst_dtm"])

        tr_feat = pd.concat([tr_cal, tr_df[weather_cols]], axis=1)
        va_feat = pd.concat([va_cal, va_df[weather_cols]], axis=1)

        feature_cols = tr_feat.columns.tolist()

        imputer = SimpleImputer(strategy="median")
        X_tr = pd.DataFrame(imputer.fit_transform(tr_feat), columns=feature_cols)
        X_va = pd.DataFrame(imputer.transform(va_feat), columns=feature_cols)

        pred_df = pd.DataFrame(index=va_df.index)

        for target in TARGET_COLS:
            target_mask = tr_df[target].notna()
            y_tr = tr_df.loc[target_mask, target]

            if target_mask.sum() == 0:
                pred_df[target] = np.nan
                continue

            model = RandomForestRegressor(
                n_estimators=120,
                max_depth=14,
                min_samples_leaf=8,
                max_features="sqrt",
                random_state=42,
                n_jobs=-1,
            )
            model.fit(X_tr.loc[target_mask], y_tr)

            pred = model.predict(X_va)
            pred_df[target] = np.clip(pred, 0, CAPACITY_KWH[target])

        answer_df = va_df[TARGET_COLS].copy().fillna(0.0)
        pred_df = pred_df.fillna(0.0)

        score, nmae_score, ficr_score = metric(answer_df, pred_df)
        fold_scores.append(
            {"fold": fold["name"], "score": score, "1-NMAE": nmae_score, "FICR": ficr_score}
        )
        print(
            f"[{fold['name']}] train={tr_mask.sum()} rows, valid={va_mask.sum()} rows "
            f"| Score={score:.4f} | 1-NMAE={nmae_score:.4f} | FICR={ficr_score:.4f}"
        )

    cv_result = pd.DataFrame(fold_scores)
    print("\n=== CV 평균 ===")
    print(cv_result[["score", "1-NMAE", "FICR"]].mean())
    return cv_result

In [ ]:
# weather_cols: train_df에 실제 존재하는 기상 피처만 (calendar 제외)
weather_cols = [c for c in train_df.columns if c not in ["forecast_kst_dtm", *TARGET_COLS]]

cv_result = run_cv(train_df, weather_cols)

[fold1_2024H1] train=17520 rows, valid=4368 rows | Score=0.5677 | 1-NMAE=0.8587 | FICR=0.2767
[fold2_2024H2] train=21888 rows, valid=4416 rows | Score=0.5855 | 1-NMAE=0.8677 | FICR=0.3033

=== CV 평균 ===
score     0.57660
1-NMAE    0.86319
FICR      0.29001
dtype: float64


## 5. 모델 최종 학습

KPX 그룹별로 Label 제공 기간이 다르므로, 각 그룹의 Label이 존재하는 행만 사용해 RandomForest 모델을 따로 학습합니다.

RandomForest는 여러 결정트리를 앙상블하는 배깅(Bagging) 모델로, 각 트리를 부트스트랩 샘플과 랜덤 피처
subset으로 학습한 뒤 예측을 평균 냅니다.


In [ ]:
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

predictions = pd.DataFrame(index=sample_submission.index)

for target in TARGET_COLS:
    train_mask = train_df[target].notna()
    y_train = train_df.loc[train_mask, target]

    model = RandomForestRegressor(
        n_estimators=120,
        max_depth=14,
        min_samples_leaf=8,
        max_features="sqrt",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train_imp.loc[train_mask], y_train)

    pred = model.predict(X_test_imp)
    pred = np.clip(pred, 0, CAPACITY_KWH[target])
    predictions[target] = pred

    print(target, "train rows:", int(train_mask.sum()))


kpx_group_1 train rows: 26200
kpx_group_2 train rows: 26201
kpx_group_3 train rows: 17538


## 6. 테스트 데이터 예측 생성

학습한 그룹별 모델로 평가 기간 전체의 발전량을 예측합니다. 예측값은 0 이상, 설비 용량 이하로 클리핑하여
물리적으로 불가능한 값을 방지합니다.


In [ ]:
submission = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
for target in TARGET_COLS:
    submission[target] = predictions[target]

submission["forecast_kst_dtm"] = pd.to_datetime(submission["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")

print(submission.head())
print(submission.shape)


     forecast_id     forecast_kst_dtm   kpx_group_1   kpx_group_2  \
0  forecast_0001  2025-01-01 01:00:00  17766.420881  18051.264933   
1  forecast_0002  2025-01-01 02:00:00  17445.286162  18005.887236   
2  forecast_0003  2025-01-01 03:00:00  15830.617570  16980.932042   
3  forecast_0004  2025-01-01 04:00:00  15158.189443  15936.857153   
4  forecast_0005  2025-01-01 05:00:00  14796.188451  15723.921137   

    kpx_group_3  
0  16309.849054  
1  15805.546254  
2  14244.983937  
3  13200.649653  
4  13451.229617  
(8760, 5)


## 7. baseline_submit.csv 생성

`sample_submission.csv` 형식에 맞춰 최종 제출 파일을 생성합니다. 이 대회는 결과 CSV를 직접 제출하는
방식이므로, 이 파일을 그대로 제출하면 됩니다.


In [ ]:
output_path = DATA_DIR / "baseline_submit.csv"
submission.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Saved: {output_path}")


Saved: /content/drive/MyDrive/비타민/풍력발전량예측/data/baseline_submit.csv
